# RetailOps: baseline Qwen trên Colab và kết nối ngrok tùy chọn
Chạy từng cell theo thứ tự, với dữ liệu giả lập. Notebook này chưa được chạy với GPU khi chuẩn bị kit.

Chọn **Runtime → Change runtime type → GPU**. Chạy baseline ngay trong notebook trước.
Tunnel chỉ dành cho phiên thử nghiệm tương tác phù hợp với tài khoản Colab; xem README và [Colab FAQ](https://research.google.com/colaboratory/faq.html).
Đây không phải server 24/7. Không có anti-idle, tự reconnect hoặc tác vụ chạy nền để giữ phiên.

File upload ở cell tiếp theo là `RetailOps_SLM_Starter.zip`. Không cần đưa token GitHub vào notebook.


In [ ]:
import io, json, os, stat, subprocess, sys, time, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload đúng một file RetailOps_SLM_Starter.zip")
archive = next(iter(uploaded.values()))
BASE = Path('/content/retailops_ec2_colab')
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    if sum(item.file_size for item in z.infolist()) > 20_000_000:
        raise ValueError("Kit bất thường: quá lớn")
    for item in z.infolist():
        p = PurePosixPath(item.filename)
        if (p.is_absolute() or '..' in p.parts or not p.parts
            or p.parts[0] != 'retailops_ec2_colab'
            or stat.S_ISLNK(item.external_attr >> 16)):
            raise ValueError("Archive path không hợp lệ")
    z.extractall('/content')
os.chdir(BASE)
(BASE / 'artifacts').mkdir(exist_ok=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
print('Source và tests đã sẵn sàng.')


## Cài Ollama và ghi thông tin GPU
Cell tải installer chính thức của Ollama. Lần bootstrap dùng phiên bản hiện hành;
`doctor` bên dưới ghi runtime version và model digest. Sau baseline đầu tiên, pin
phiên bản runtime đã chạy thành công để so sánh các lần tiếp theo.

Nếu cell này không tìm thấy GPU, đổi runtime trước. Có GPU chưa chứng minh Ollama đã
đưa toàn bộ model lên GPU; kiểm tra `ollama ps` sau lượt inference đầu.


In [ ]:
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     text=True, capture_output=True, check=True).stdout
print(gpu)
(BASE / 'artifacts' / 'gpu.txt').write_text(gpu, encoding='utf-8')
installer = Path('/tmp/retailops-ollama-install.sh')
subprocess.run(['curl', '-fsSL', 'https://ollama.com/install.sh', '-o', str(installer)], check=True)
subprocess.run(['sh', str(installer)], check=True)
MODEL = 'qwen3.5:4b'
OLLAMA_ENV = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_NO_CLOUD='1',
                  OLLAMA_NUM_PARALLEL='1', OLLAMA_MAX_LOADED_MODELS='1', RETAILOPS_MODEL=MODEL)
if 'ollama_process' in globals() and ollama_process.poll() is None:
    raise RuntimeError('Ollama của notebook vẫn đang chạy; không khởi chạy bản thứ hai.')
ollama_log = open('/tmp/retailops-ollama.log', 'w')
ollama_process = subprocess.Popen(['ollama', 'serve'], env=OLLAMA_ENV,
                                  stdout=ollama_log, stderr=subprocess.STDOUT)
import urllib.request
local_http = urllib.request.build_opener(urllib.request.ProxyHandler({}))
for _ in range(60):
    if ollama_process.poll() is not None:
        raise RuntimeError('Ollama không khởi động được. Kiểm tra /tmp/retailops-ollama.log; có thể port đang được dùng.')
    try:
        with local_http.open('http://127.0.0.1:11434/api/version', timeout=2) as response:
            json.load(response)
        break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError('Ollama chưa sẵn sàng; kiểm tra log.')
subprocess.run(['ollama', 'pull', MODEL], env=OLLAMA_ENV, check=True)


## Chạy baseline ngay trong notebook
24 tình huống chỉ là smoke test. Evaluation bỏ qua extraction cache.
Latency bao gồm việc nạp model nếu xảy ra. Không gọi đây là điểm benchmark độc lập.
Mẫu đều phải là dữ liệu giả lập vì starter lưu nguyên văn request/output.


In [ ]:
identity_text = subprocess.check_output(
    [sys.executable, 'retailops_baseline.py', '--model', MODEL, 'doctor'], text=True)
(BASE / 'artifacts' / 'model-identity.json').write_text(identity_text, encoding='utf-8')
print(identity_text)
subprocess.run([sys.executable, 'retailops_baseline.py', '--model', MODEL,
                'evaluate', '--cases', 'data/smoke.jsonl'], check=True)
subprocess.run(['ollama', 'ps'], env=OLLAMA_ENV, check=True)


## Tùy chọn: bật proxy và ngrok trong phiên thử nghiệm
Chỉ thực hiện nếu cách dùng phù hợp với tài khoản của bạn; đổi `ENABLE_REMOTE_EXPERIMENT` thành `True`.
Trong Colab Secrets, tạo hai mục và cấp quyền cho notebook:

- `NGROK_AUTHTOKEN`: từ tài khoản ngrok.
- `RETAILOPS_INFERENCE_TOKEN`: chuỗi ngẫu nhiên URL-safe ít nhất 32 ký tự, dùng chung với EC2.

Tạo token trong password manager hoặc trên máy riêng bằng `secrets.token_urlsafe(32)`.
Không in token vào output notebook. API chỉ expose proxy, không expose Ollama.
Ngrok request inspection được tắt trong cấu hình tunnel. Không đưa dữ liệu thật qua kit này.


In [ ]:
ENABLE_REMOTE_EXPERIMENT = False
if ENABLE_REMOTE_EXPERIMENT:
    from google.colab import userdata
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyngrok>=7,<8'], check=True)
    from pyngrok import ngrok
    import re
    from urllib.parse import urlparse
    token = userdata.get('RETAILOPS_INFERENCE_TOKEN')
    if not re.fullmatch(r'[A-Za-z0-9_-]{32,128}', token):
        raise ValueError('Inference token phải có 32–128 ký tự URL-safe')
    if 'proxy_process' in globals() and proxy_process.poll() is None:
        raise RuntimeError('Proxy đã chạy; dừng phiên cũ trước khi khởi chạy lại.')
    proxy_env = dict(OLLAMA_ENV, RETAILOPS_INFERENCE_TOKEN=token)
    proxy_log = open('/tmp/retailops-proxy.log', 'w')
    proxy_process = subprocess.Popen([sys.executable, 'inference_proxy.py'], env=proxy_env,
                                     stdout=proxy_log, stderr=subprocess.STDOUT)
    for _ in range(40):
        if proxy_process.poll() is not None:
            raise RuntimeError('Proxy không khởi động được; kiểm tra log trong /tmp.')
        try:
            req = urllib.request.Request('http://127.0.0.1:8001/healthz',
                                          headers={'Authorization': 'Bearer ' + token})
            with local_http.open(req, timeout=3) as response:
                json.load(response)
            break
        except OSError:
            time.sleep(0.5)
    else:
        raise RuntimeError('Proxy/model chưa sẵn sàng; chưa mở tunnel.')
    ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
    tunnel = ngrok.connect(addr='http://127.0.0.1:8001', proto='http', bind_tls=True, inspect=False)
    if not tunnel.public_url.startswith('https://'):
        ngrok.disconnect(tunnel.public_url)
        raise RuntimeError('Cần HTTPS endpoint')
    print('RETAILOPS_MODEL_URL=' + tunnel.public_url)
    print('RETAILOPS_ALLOWED_HOST=' + urlparse(tunnel.public_url).hostname)
    print('Điền hai giá trị trên và cùng inference token vào inference.env trên EC2.')
    packages = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    (BASE / 'artifacts' / 'runtime-requirements.txt').write_text(packages, encoding='utf-8')
else:
    print('Chỉ chạy baseline trong notebook; tunnel chưa được bật.')


## Tải kết quả trước khi kết thúc phiên
Các lần gọi từ EC2 lưu log trên EC2. Cell này tải log/report từ các lượt baseline
chạy trực tiếp trong Colab. Dùng SQLite backup để giữ các bản ghi đã commit trong WAL;
không copy riêng file database khi có tiến trình đang ghi.


In [ ]:
from backup_state import backup
import shutil, uuid
export_dir = Path('/content') / ('retailops-results-' + uuid.uuid4().hex[:8])
export_dir.mkdir()
source_db = BASE / 'artifacts' / 'runs.sqlite3'
if source_db.exists():
    backup(source_db, export_dir / 'runs.sqlite3')
for pattern in ('report-*.json', 'model-identity.json', 'gpu.txt', 'runtime-requirements.txt'):
    for source in (BASE / 'artifacts').glob(pattern):
        shutil.copy2(source, export_dir / source.name)
result_zip = shutil.make_archive(str(export_dir), 'zip', export_dir)
files.download(result_zip)


## Dừng sau phiên thử nghiệm
Đóng tunnel, proxy và model server do notebook này khởi chạy. Khi hoàn tất, dùng
Colab **Disconnect and delete runtime** để kết thúc compute. Token sẽ không được in ra.


In [ ]:
if 'tunnel' in globals():
    ngrok.disconnect(tunnel.public_url)
for process_name in ('proxy_process', 'ollama_process'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)
print('Đã dừng các tiến trình của phiên notebook.')
